# RESuM controlled experiments

Choose a YAML experiment, inspect its settings, run it, and browse saved results. Model and evaluation code lives in the package. See `docs/experiments.md` for configuration and statistical interpretation.


In [ ]:
from pathlib import Path
import json
import torch
from IPython.display import display, HTML, JSON, Image
from schemas.experiments import load_experiment
from core.experiments.runner import validate_experiment, run
from core.experiments.storage import RunStore
from core.experiments.comparison import compare
from core.experiments.reporting import regenerate

torch.set_num_threads(1)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/optical_fine_tuning.yaml").exists())
CONFIG = ROOT / "experiments/optical_fine_tuning.yaml"
RUN_EXPERIMENT = False
spec = validate_experiment(CONFIG)
store = RunStore(spec.store)
display(JSON(spec.model_dump(mode="json")))


## Launch or inspect history

Each launch creates a new run. It does not overwrite previous results. Select an explicit run ID below to review an older experiment.


In [ ]:
output = run(spec) if RUN_EXPERIMENT else None
history = store.list()
display(JSON([{k: r.get(k) for k in ("id", "name", "kind", "status", "created_at", "parents")} for r in history]))


In [ ]:
RUN_ID = output.name if output else None  # Or paste a saved run ID.
if RUN_ID:
    record = store.inspect(RUN_ID)
    display(JSON(record))
    run_path = Path(record["path"])
    if (run_path / "metrics.json").exists():
        display(JSON(json.loads((run_path / "metrics.json").read_text())))
    report = run_path / "report"
    if report.exists():
        for image_path in sorted(report.rglob("*.png")):
            display(Image(filename=str(image_path)))


## Controlled comparison

The configuration lists checkpoint/model overrides and paired seeds. Both variants use the same immutable dataset and evaluation targets. The example fits MFGPs from saved checkpoints without retraining the CNP.


In [ ]:
COMPARISON = ROOT / "experiments/optical_mfgp_comparison.yaml"
RUN_COMPARISON = False
if RUN_COMPARISON:
    comparison = compare(COMPARISON)
    display(JSON(json.loads((comparison / "summary.json").read_text())))


## Regenerate a report

This uses saved prediction arrays and creates a separate report directory. It does not retrain or change the completed run.


In [ ]:
REGENERATE_REPORT = False
if REGENERATE_REPORT and RUN_ID:
    report = regenerate(spec.store, RUN_ID)
    print(report / "index.html")
